# 03 ? PDP Sizing Analysis: Availability vs Conversion
### E-Commerce Product Analytics ? Search & Conversion Funnel
**Objective:** Investigate how size stockouts on Product Display Pages (PDP) affect Add-to-Cart Rates (ATCR) across apparel and footwear categories.

---
### Core Hypotheses:
- Out-of-stock sizes correlate with severe consideration drop-off.
- Sizing friction is concentrated in multi-size apparel categories (Dresses, Tops, Ethnic Wear).
- High dwell time on out-of-stock PDPs reflects user hesitation rather than healthy engagement.


In [ ]:
import sys
import os
sys.path.append('../src')
import duckdb
import pandas as pd
import numpy as np
from exploratory_analysis import get_db_connection, two_proportion_z_test, calc_odds_ratio

con = get_db_connection()


## 1. Overall Size Availability vs Add-to-Cart Rate (ATCR)


In [ ]:
q_stock = '''
SELECT 
    is_size_in_stock,
    COUNT(*) AS views,
    SUM(CASE WHEN added_to_cart THEN 1 ELSE 0 END) AS added_carts,
    ROUND(100.0 * SUM(CASE WHEN added_to_cart THEN 1 ELSE 0 END) / COUNT(*), 2) AS atcr_pct
FROM product_views
GROUP BY 1
ORDER BY 1 DESC
'''
df_stock = con.execute(q_stock).df()
print("In-Stock vs Out-of-Stock ATCR:")
print(df_stock.to_string())

in_cart, in_views = df_stock.loc[df_stock['is_size_in_stock']==True, ['added_carts', 'views']].values[0]
out_cart, out_views = df_stock.loc[df_stock['is_size_in_stock']==False, ['added_carts', 'views']].values[0]

diff, z, p, ci = two_proportion_z_test(in_cart, in_views, out_cart, out_views)
odds_r, ci_low, ci_high = calc_odds_ratio(in_cart, in_views, out_cart, out_views)

print(f"\nStatistical Validation:")
print(f"Absolute Difference: +{diff*100:.2f} pp [95% CI: {ci[0]*100:.2f} pp, {ci[1]*100:.2f} pp]")
print(f"Relative Difference: {((in_cart/in_views)/(out_cart/out_views) - 1)*100:.1f}%")
print(f"Odds Ratio: {odds_r:.2f} [95% CI: {ci_low:.2f}, {ci_high:.2f}]")
print(f"Two-proportion Z-statistic: {z:.2f} (p-value: {p:.2e})")


## 2. Category-Level Stockout Impact
Analyzing whether the sizing stockout penalty is universal across sub-categories.


In [ ]:
q_cat = '''
SELECT 
    p.sub_category,
    COUNT(*) AS total_views,
    ROUND(100.0 * SUM(CASE WHEN pv.is_size_in_stock THEN 1 ELSE 0 END) / COUNT(*), 2) AS stock_avail_pct,
    ROUND(100.0 * SUM(CASE WHEN pv.is_size_in_stock AND pv.added_to_cart THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN pv.is_size_in_stock THEN 1 ELSE 0 END), 0), 2) AS instock_atcr,
    ROUND(100.0 * SUM(CASE WHEN NOT pv.is_size_in_stock AND pv.added_to_cart THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN NOT pv.is_size_in_stock THEN 1 ELSE 0 END), 0), 2) AS outstock_atcr
FROM product_views pv
JOIN products p ON pv.product_id = p.product_id
GROUP BY 1
HAVING SUM(CASE WHEN NOT pv.is_size_in_stock THEN 1 ELSE 0 END) > 0
ORDER BY total_views DESC
'''
df_cat = con.execute(q_cat).df()
print("Category Sizing Stockout Impact:")
print(df_cat.to_string())


## 3. Dwell Time vs Conversion Curve


In [ ]:
q_dwell = '''
SELECT 
    CASE 
        WHEN dwell_time_sec < 15 THEN '1. <15s (Bounce)'
        WHEN dwell_time_sec BETWEEN 15 AND 30 THEN '2. 15-30s (Skim)'
        WHEN dwell_time_sec BETWEEN 31 AND 60 THEN '3. 31-60s (Consideration)'
        WHEN dwell_time_sec BETWEEN 61 AND 120 THEN '4. 61-120s (Engaged)'
        ELSE '5. >120s (Hesitation)'
    END AS dwell_bucket,
    COUNT(*) AS views,
    ROUND(100.0 * SUM(CASE WHEN added_to_cart THEN 1 ELSE 0 END) / COUNT(*), 2) AS atcr_pct
FROM product_views
GROUP BY 1
ORDER BY 1
'''
df_dwell = con.execute(q_dwell).df()
print("Dwell Time vs Add-to-Cart Conversion:")
print(df_dwell.to_string())


## 4. Visual Evidence


In [ ]:
from IPython.display import Image, display
display(Image(filename='../reports/figures/05_instock_vs_outstock_atcr.png'))
display(Image(filename='../reports/figures/06_atcr_vs_dwell_time.png'))
display(Image(filename='../reports/figures/07_category_traffic_vs_atcr.png'))
